# Identify new indications in the latest label

This notebook runs the new-indication discovery path: extract indications from the latest label, map them against existing MOAlmanac indications, and retain only latest-label records classified as `new`.

Revision assessment, removal review, new-indication curation, and approval-date matching are intentionally out of scope here.

In [ ]:
import json
import os
from collections import Counter
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

from moalmanac_fda_curation.core.check_curation_preflight import check_curation_preflight
from moalmanac_fda_curation.core.extract_indications_from_fda_label import (
    DEFAULT_MAX_TOKENS,
    DEFAULT_MODEL,
    extract_indications_from_label_url,
)
from moalmanac_fda_curation.core.identify_new_indications import (
    build_indication_mapping_prompt,
    indexed_latest_indications,
    map_existing_to_latest_indications,
    select_new_indication_candidates,
)
from moalmanac_fda_curation.core.moalmanac_records import load_existing_indications

## Configure the Opdivo comparison

The curation-status check finds the existing MOAlmanac document and the newest approved FDA label. The existing indications come from the neighboring `moalmanac-db` checkout.

In [ ]:
env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("No .env file found")
load_dotenv(env_path)
if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError(f"ANTHROPIC_API_KEY is missing from {env_path}")

APPLICATION_NUMBER = "BLA125554"
PROJECT_ROOT = Path(env_path).parent
DATABASE_DIR = PROJECT_ROOT.parent / "moalmanac-db"
EXTRACTION_DIR = PROJECT_ROOT / "analyses/new-indication-inputs/opdivo-current-label"

status = check_curation_preflight(APPLICATION_NUMBER, DATABASE_DIR)
if not status["previously_curated"]:
    raise RuntimeError(f"{APPLICATION_NUMBER} is not curated in {DATABASE_DIR}")
DOCUMENT_ID = status["document_id"]
LATEST_LABEL_URL = status["latest_label_url"]
print("MOAlmanac document:", DOCUMENT_ID)
print("Latest label date:", status["latest_label_date"])
print("Latest label URL:", LATEST_LABEL_URL)

## Optionally regenerate latest-label indications

Leave `REGENERATE = False` to evaluate mapping against the saved extraction. Set it to `True` only when you want to download the current label and rerun the indication-extraction LLM call.

In [ ]:
REGENERATE = False

if REGENERATE:
    extraction_paths = extract_indications_from_label_url(
        label_url=LATEST_LABEL_URL,
        brand_name="Opdivo",
        generic_name="nivolumab",
        application_number=APPLICATION_NUMBER,
        labels_dir=EXTRACTION_DIR / "labels",
        indications_dir=EXTRACTION_DIR / "intermediate",
        model=DEFAULT_MODEL,
        max_tokens=DEFAULT_MAX_TOKENS,
        overwrite=True,
    )
else:
    extraction_paths = {
        "claude_chunked_indication_fields": (
            EXTRACTION_DIR
            / "intermediate/Opdivo-BLA125554-claude_chunked_indication_fields.json"
        )
    }

LATEST_EXTRACTION_JSON = extraction_paths["claude_chunked_indication_fields"]
if not LATEST_EXTRACTION_JSON.is_file():
    raise FileNotFoundError(
        f"No saved extraction at {LATEST_EXTRACTION_JSON}; set REGENERATE = True to create it."
    )
print("Extraction artifact:", LATEST_EXTRACTION_JSON)

## Load and inspect the indication strings

Only indication IDs, extraction indexes, and indication strings are supplied to the mapping call. Existing indications are the document's current (`Approved` or `Accelerated`) records in `moalmanac-db`. Biomarker scope is applied afterward, when selecting `new` candidates.

In [ ]:
existing_indications = load_existing_indications(DATABASE_DIR, DOCUMENT_ID)
latest_payload = json.loads(LATEST_EXTRACTION_JSON.read_text())
latest_indications = indexed_latest_indications(latest_payload)

print(f"Existing indications: {len(existing_indications)}")
for indication in existing_indications:
    print(f"  {indication['id']}: {indication['indication']}")

print(f"\nLatest-label indications: {len(latest_indications)}")
for indication in latest_indications:
    print(
        f"  [{indication['latest_indication_index']}]: "
        f"{indication['indication']}"
    )

## Inspect the exact prompt

Read this before running the LLM. It is the complete identity-mapping prompt.

In [ ]:
mapping_prompt = build_indication_mapping_prompt(
    existing_indications, latest_indications
)
print(mapping_prompt)

## Run one mapping call

This is the notebook's only LLM evaluation step. Verification checks that every indication is accounted for exactly once.

In [ ]:
mapping_result = map_existing_to_latest_indications(
    existing_indications, latest_indications
)

print("Verified:", mapping_result["verified"])
print("Verification errors:", mapping_result["verification_errors"])
print(
    "Classification counts:",
    Counter(item["classification"] for item in mapping_result["mappings"]),
)

## Select new-indication candidates

This selection is deterministic: it retains the complete latest-label extraction record from each verified mapping classified as `new` that names a biomarker.

In [ ]:
new_candidates = select_new_indication_candidates(mapping_result)
print(f"New-indication candidates: {len(new_candidates)}")
for number, candidate in enumerate(new_candidates, start=1):
    print("=" * 100)
    print(f"CANDIDATE {number}")
    print("Latest index:", candidate["latest_indication_index"])
    print("Indication:", candidate["indication"])
    print("Biomarkers:", candidate.get("raw_biomarkers"))
    print("Cancer types:", candidate.get("raw_cancer_types"))
    print("Therapeutics:", candidate.get("raw_therapeutics"))
    print("Reason:", candidate["new_candidate_reason"])

## Optionally save this run

In [ ]:
SAVE_RESULTS = False
RESULTS_JSON = EXTRACTION_DIR / "opdivo-new-indication-candidates.json"

if SAVE_RESULTS:
    RESULTS_JSON.write_text(json.dumps(new_candidates, indent=2) + "\n")
    print("Saved:", RESULTS_JSON)